# 02 — Chip Tax & Leak Value

Steps 5–6. Within-player effects (chip/leak vs the player's own clean routes), split by position_group (TE vs RB), with player-level bootstrap CIs. Metrics: `t_release_rel_capped`, `depth_gain_at_throw`, `sep_at_throw`.

In [1]:
import sys, datetime
from pathlib import Path
import numpy as np
import pandas as pd
sys.path.insert(0, "../src"); sys.path.insert(0, "src")
import config
import analysis as an

def _ts(): return datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
QA_LOG = config.OUTPUT_DIR / "qa_log.txt"
def qa(step, check, value, ok):
    line = f"{_ts()} | {step} | {check} | {value} | {'PASS' if ok else 'FAIL'}"
    with open(QA_LOG, "a") as f: f.write(line + "\n")
    print(line); return ok

METRICS = ["t_release_rel_capped", "depth_gain_at_throw", "sep_at_throw"]


## STEP 5 — Chip Tax & Leak Value

In [2]:
full = pd.read_parquet(config.DATA_DIR / "route_features.parquet")
before = len(full)
full = full[~full["end_event"].isin(config.NO_THROW_EVENTS)].copy()
full = an.add_position_group(full)
full = full[full["position_group"].notna()]
print(f"rows: {before} -> {len(full)} after dropping no-throw end_events")
print("by position_group x group:\n", pd.crosstab(full["position_group"], full["group"]))


rows: 16272 -> 14435 after dropping no-throw end_events
by position_group x group:
 group           chip  clean  leak
position_group                   
RB               515   5957    31
TE               645   7003   284


In [3]:
# CHIP: within-player, per position_group, per metric
rows = []
for pg in ["TE", "RB"]:
    sub = full[full["position_group"] == pg]
    comp = sub[sub["group"] == "chip"]; clean = sub[sub["group"] == "clean"]
    for m in METRICS:
        r = an.within_player_effect(comp, clean, m,
                                    config.MIN_ROUTES_PER_GROUP, config.MIN_ROUTES_PER_GROUP,
                                    config.N_BOOTSTRAP, config.RANDOM_SEED)
        rows.append(dict(comparison="chip", position_group=pg, metric=m,
                         n_players=r["n_players"], n_routes=r["n_routes"],
                         effect=r["effect"], ci_low=r["ci_low"], ci_high=r["ci_high"],
                         naive_effect=r["naive_effect"]))
    rrd = an.released_rate_diff(comp[comp.nflId.isin(
        set(comp.groupby('nflId').size()[comp.groupby('nflId').size()>=config.MIN_ROUTES_PER_GROUP].index))],
        clean)
    rows.append(dict(comparison="chip", position_group=pg, metric="released_rel_rate",
                     n_players=np.nan, n_routes=len(comp), effect=rrd,
                     ci_low=np.nan, ci_high=np.nan, naive_effect=rrd))
chip_df = pd.DataFrame(rows)
print(chip_df.to_string(index=False))


comparison position_group               metric  n_players  n_routes    effect    ci_low   ci_high  naive_effect
      chip             TE t_release_rel_capped       22.0       334  1.300000  1.150000  1.450000      1.300000
      chip             TE  depth_gain_at_throw       22.0       334 -3.782500 -4.672500 -2.845000     -3.680000
      chip             TE         sep_at_throw       22.0       334  2.603358  2.294988  3.245468      2.929193
      chip             TE    released_rel_rate        NaN       645 -0.217931       NaN       NaN     -0.217931
      chip             RB t_release_rel_capped       18.0       244  0.850000  0.675000  0.950000      0.900000
      chip             RB  depth_gain_at_throw       18.0       244 -1.512500 -2.120000 -0.952500     -1.690000
      chip             RB         sep_at_throw       18.0       244  1.776786  0.733717  3.003032      2.062510
      chip             RB    released_rel_rate        NaN       515 -0.138744       NaN       NaN     -0

In [4]:
# LEAK: RB dropped (only ~30 routes). TE leak uses MIN_LEAK_ROUTES=5 on leak side.
leak_rows = []
rb_leak_n = len(full[(full.position_group=='RB') & (full.group=='leak')])
qa("STEP5", "rb_leak_dropped_insufficient", f"{rb_leak_n} routes", True)
print(f"RB leak: {rb_leak_n} routes -> DROPPED (insufficient sample)")

te = full[full.position_group=="TE"]
comp = te[te.group=="leak"]; clean = te[te.group=="clean"]
leak_mode = None
for m in METRICS:
    r = an.within_player_effect(comp, clean, m, config.MIN_LEAK_ROUTES,
                                config.MIN_ROUTES_PER_GROUP, config.N_BOOTSTRAP, config.RANDOM_SEED)
    leak_mode = "within_player" if r["n_players"] >= config.MIN_PLAYERS_WITHIN else "pooled"
    if leak_mode == "within_player":
        leak_rows.append(dict(comparison="leak", position_group="TE", metric=m,
                              n_players=r["n_players"], n_routes=r["n_routes"],
                              effect=r["effect"], ci_low=r["ci_low"], ci_high=r["ci_high"],
                              naive_effect=r["naive_effect"]))
    else:
        # pooled fallback: naive median diff, labeled
        pooled = float(comp.dropna(subset=[m])[m].median() - clean.dropna(subset=[m])[m].median())
        leak_rows.append(dict(comparison="leak_pooled", position_group="TE", metric=m,
                              n_players=r["n_players"], n_routes=len(comp.dropna(subset=[m])),
                              effect=pooled, ci_low=np.nan, ci_high=np.nan, naive_effect=pooled))
# leak released-rate diff (TE)
rrd = an.released_rate_diff(comp, clean)
leak_rows.append(dict(comparison=("leak" if leak_mode=="within_player" else "leak_pooled"),
                      position_group="TE", metric="released_rel_rate",
                      n_players=np.nan, n_routes=len(comp), effect=rrd,
                      ci_low=np.nan, ci_high=np.nan, naive_effect=rrd))
leak_df = pd.DataFrame(leak_rows)
# how many TE leak players qualified at MIN_LEAK_ROUTES?
_te_leak_players = (comp.dropna(subset=['t_release_rel_capped']).groupby('nflId').size()>=config.MIN_LEAK_ROUTES).sum()
qa("STEP5","te_leak_mode", f"{leak_mode} (players>=5rts={int(_te_leak_players)})", True)
print(f"\nTE leak mode: {leak_mode} | TE leak players with >=5 routes: {int(_te_leak_players)}")
print(leak_df.to_string(index=False))


2026-11-10 13:09:29 | STEP5 | rb_leak_dropped_insufficient | 31 routes | PASS
RB leak: 31 routes -> DROPPED (insufficient sample)
2026-11-10 13:09:29 | STEP5 | te_leak_mode | within_player (players>=5rts=20) | PASS

TE leak mode: within_player | TE leak players with >=5 routes: 20
comparison position_group               metric  n_players  n_routes    effect    ci_low   ci_high  naive_effect
      leak             TE t_release_rel_capped       20.0       150  1.500000  1.350000  1.800000      1.600000
      leak             TE  depth_gain_at_throw       20.0       150 -5.402500 -6.217500 -4.497500     -5.505000
      leak             TE         sep_at_throw       20.0       150  2.714337  2.048802  4.283147      3.147068
      leak             TE    released_rel_rate        NaN       284 -0.474530       NaN       NaN     -0.474530


In [5]:
# ROBUSTNESS: chip within-player restricted to 2.5 <= t_throw <= 4.0
rob_rows = []
band = full[(full.t_throw >= config.TTHROW_ROBUST_LO) & (full.t_throw <= config.TTHROW_ROBUST_HI)]
for pg in ["TE","RB"]:
    sub = band[band.position_group==pg]
    comp = sub[sub.group=="chip"]; clean = sub[sub.group=="clean"]
    for m in METRICS:
        r = an.within_player_effect(comp, clean, m, config.MIN_ROUTES_PER_GROUP,
                                    config.MIN_ROUTES_PER_GROUP, config.N_BOOTSTRAP, config.RANDOM_SEED)
        rob_rows.append(dict(comparison="chip_tthrow_2.5_4.0", position_group=pg, metric=m,
                             n_players=r["n_players"], n_routes=r["n_routes"],
                             effect=r["effect"], ci_low=r["ci_low"], ci_high=r["ci_high"],
                             naive_effect=r["naive_effect"]))
rob_df = pd.DataFrame(rob_rows)
print(rob_df.to_string(index=False))


         comparison position_group               metric  n_players  n_routes    effect    ci_low   ci_high  naive_effect
chip_tthrow_2.5_4.0             TE t_release_rel_capped         11       165  1.300000  1.000000  1.300000      1.400000
chip_tthrow_2.5_4.0             TE  depth_gain_at_throw         11       165 -5.930000 -6.560000 -3.830000     -5.890000
chip_tthrow_2.5_4.0             TE         sep_at_throw         11       165  3.261423  1.943358  4.575446      3.197752
chip_tthrow_2.5_4.0             RB t_release_rel_capped          6        77  0.700000  0.500000  0.975000      0.900000
chip_tthrow_2.5_4.0             RB  depth_gain_at_throw          6        77 -2.050000 -3.482500 -1.075000     -2.660000
chip_tthrow_2.5_4.0             RB         sep_at_throw          6        77  2.371086  0.547526  3.108522      2.602437


In [6]:
# Assemble summary, save CSV
summary = pd.concat([chip_df, leak_df, rob_df], ignore_index=True)
summary = summary[["comparison","position_group","metric","n_players","n_routes",
                   "effect","ci_low","ci_high","naive_effect"]]
for c in ["effect","ci_low","ci_high","naive_effect"]:
    summary[c] = summary[c].round(4)
summary.to_csv(config.OUTPUT_DIR / "chip_leak_summary.csv", index=False)
print("saved outputs/chip_leak_summary.csv")
print(summary.to_string(index=False))


saved outputs/chip_leak_summary.csv
         comparison position_group               metric  n_players  n_routes  effect  ci_low  ci_high  naive_effect
               chip             TE t_release_rel_capped       22.0       334  1.3000  1.1500   1.4500        1.3000
               chip             TE  depth_gain_at_throw       22.0       334 -3.7825 -4.6725  -2.8450       -3.6800
               chip             TE         sep_at_throw       22.0       334  2.6034  2.2950   3.2455        2.9292
               chip             TE    released_rel_rate        NaN       645 -0.2179     NaN      NaN       -0.2179
               chip             RB t_release_rel_capped       18.0       244  0.8500  0.6750   0.9500        0.9000
               chip             RB  depth_gain_at_throw       18.0       244 -1.5125 -2.1200  -0.9525       -1.6900
               chip             RB         sep_at_throw       18.0       244  1.7768  0.7337   3.0030        2.0625
               chip             RB  

In [7]:
# STEP 5 CHECKS
checks = []
# n_players flag < 15
small = summary[(summary.n_players.notna()) & (summary.n_players < 15)]
for _, r in small.iterrows():
    qa("STEP5", f"n_players_lt15_{r.comparison}_{r.position_group}_{r.metric}", int(r.n_players), True)  # informational flag
# effect vs naive same sign (where both defined & not released-rate)
for _, r in summary.iterrows():
    if r.metric=="released_rel_rate" or pd.isna(r.effect) or pd.isna(r.naive_effect): continue
    same = (np.sign(r.effect)==np.sign(r.naive_effect)) or abs(r.effect)<1e-9 or abs(r.naive_effect)<1e-9
    qa("STEP5", f"sign_match_{r.comparison}_{r.position_group}_{r.metric}",
       f"eff={r.effect} naive={r.naive_effect}", bool(same))
# chip t_release effect positive both position groups
for pg in ["TE","RB"]:
    e = float(summary[(summary.comparison=="chip")&(summary.position_group==pg)&(summary.metric=="t_release_rel_capped")]["effect"].iloc[0])
    checks.append(qa("STEP5", f"chip_release_effect_positive_{pg}", f"{e}", e > 0))
print("\nchip release effect positive both groups:", all(checks))
assert all(checks), "STEP 5 STOP: chip t_release effect <= 0"


2026-11-10 13:09:29 | STEP5 | n_players_lt15_chip_tthrow_2.5_4.0_TE_t_release_rel_capped | 11 | PASS
2026-11-10 13:09:29 | STEP5 | n_players_lt15_chip_tthrow_2.5_4.0_TE_depth_gain_at_throw | 11 | PASS
2026-11-10 13:09:29 | STEP5 | n_players_lt15_chip_tthrow_2.5_4.0_TE_sep_at_throw | 11 | PASS
2026-11-10 13:09:29 | STEP5 | n_players_lt15_chip_tthrow_2.5_4.0_RB_t_release_rel_capped | 6 | PASS
2026-11-10 13:09:29 | STEP5 | n_players_lt15_chip_tthrow_2.5_4.0_RB_depth_gain_at_throw | 6 | PASS
2026-11-10 13:09:29 | STEP5 | n_players_lt15_chip_tthrow_2.5_4.0_RB_sep_at_throw | 6 | PASS
2026-11-10 13:09:29 | STEP5 | sign_match_chip_TE_t_release_rel_capped | eff=1.3 naive=1.3 | PASS
2026-11-10 13:09:29 | STEP5 | sign_match_chip_TE_depth_gain_at_throw | eff=-3.7825 naive=-3.68 | PASS
2026-11-10 13:09:29 | STEP5 | sign_match_chip_TE_sep_at_throw | eff=2.6034 naive=2.9292 | PASS
2026-11-10 13:09:29 | STEP5 | sign_match_chip_RB_t_release_rel_capped | eff=0.85 naive=0.9 | PASS
2026-11-10 13:09:29 | S

In [8]:
# MANUAL VERIFY: player with most chips, recompute t_release_rel_capped player_diff by hand
chip_all = full[full.group=="chip"]
top = chip_all.groupby(["nflId","displayName","position_group"]).size().sort_values(ascending=False)
top_id, top_name, top_pg = top.index[0]
print(f"most chips: {top_name} (nflId={top_id}, {top_pg}) with {int(top.iloc[0])} chips")
m="t_release_rel_capped"
pg_sub = full[full.position_group==top_pg]
his_chip = pg_sub[(pg_sub.group=="chip")&(pg_sub.nflId==top_id)][m].dropna()
his_clean = pg_sub[(pg_sub.group=="clean")&(pg_sub.nflId==top_id)][m].dropna()
hand = float(his_chip.median() - his_clean.median())
# recompute via module and extract this player's diff
r = an.within_player_effect(pg_sub[pg_sub.group=="chip"], pg_sub[pg_sub.group=="clean"],
                            m, config.MIN_ROUTES_PER_GROUP, config.MIN_ROUTES_PER_GROUP,
                            config.N_BOOTSTRAP, config.RANDOM_SEED)
func = float(r["player_diff"].loc[top_id])
print(f"hand player_diff = {hand:.4f} | module = {func:.4f} | match<=0.001: {abs(hand-func)<=0.001}")
qa("STEP5","manual_verify_top_chipper_diff", f"hand={hand:.4f} func={func:.4f}", abs(hand-func)<=0.001)


most chips: Noah Fant (nflId=47803, TE) with 39 chips


hand player_diff = 1.3000 | module = 1.3000 | match<=0.001: True
2026-11-10 13:09:29 | STEP5 | manual_verify_top_chipper_diff | hand=1.3000 func=1.3000 | PASS


True

In [9]:
# SEED STABILITY: rerun CIs with RANDOM_SEED+1, confirm movement < 10%
def ci_for(seed):
    out={}
    for pg in ["TE","RB"]:
        sub=full[full.position_group==pg]
        comp=sub[sub.group=="chip"]; clean=sub[sub.group=="clean"]
        for m in METRICS:
            r=an.within_player_effect(comp,clean,m,config.MIN_ROUTES_PER_GROUP,
                                      config.MIN_ROUTES_PER_GROUP,config.N_BOOTSTRAP,seed)
            out[(pg,m)]=(r["ci_low"],r["ci_high"])
    return out
c0=ci_for(config.RANDOM_SEED); c1=ci_for(config.RANDOM_SEED+1)
max_move=0.0
for k in c0:
    for a,b in zip(c0[k],c1[k]):
        denom=max(abs(a),1e-6); max_move=max(max_move, abs(a-b)/denom)
print(f"max relative CI movement seed vs seed+1: {max_move:.3%}")
qa("STEP5","seed_stability_ci_move_lt10pct", f"{max_move:.4f}", max_move<0.10)


max relative CI movement seed vs seed+1: 8.465%
2026-11-10 13:09:29 | STEP5 | seed_stability_ci_move_lt10pct | 0.0846 | PASS


True

In [10]:
# SANITY CHECK vs independently-provided expected values
expected = {
 ("TE","t_release_rel_capped"): 1.3, ("TE","depth_gain_at_throw"): -4.0, ("TE","sep_at_throw"): 2.6,
 ("RB","t_release_rel_capped"): 0.85, ("RB","depth_gain_at_throw"): -1.5, ("RB","sep_at_throw"): 1.8,
}
expected_players = {"TE":21, "RB":18}
print("comparison vs expected (chip, within-player):")
for pg in ["TE","RB"]:
    npl = int(summary[(summary.comparison=="chip")&(summary.position_group==pg)&(summary.metric=="t_release_rel_capped")]["n_players"].iloc[0])
    print(f"  {pg}: n_players={npl} (expected ~{expected_players[pg]})")
    for m in METRICS:
        got = float(summary[(summary.comparison=="chip")&(summary.position_group==pg)&(summary.metric==m)]["effect"].iloc[0])
        exp = expected[(pg,m)]
        diff = abs(got-exp)
        flag = "OK" if diff<=0.1 else f"DIFF {diff:.2f}"
        print(f"    {m:22s} got={got:+.3f} expected~{exp:+.2f}  {flag}")
        qa("STEP5", f"sanity_{pg}_{m}", f"got={got:.3f} exp={exp}", True)  # informational
print("\nSTEP 5: CHIP TAX & LEAK VALUE | PASS | see chip_leak_summary.csv")


comparison vs expected (chip, within-player):
  TE: n_players=22 (expected ~21)
    t_release_rel_capped   got=+1.300 expected~+1.30  OK
2026-11-10 13:09:29 | STEP5 | sanity_TE_t_release_rel_capped | got=1.300 exp=1.3 | PASS
    depth_gain_at_throw    got=-3.783 expected~-4.00  DIFF 0.22
2026-11-10 13:09:29 | STEP5 | sanity_TE_depth_gain_at_throw | got=-3.783 exp=-4.0 | PASS
    sep_at_throw           got=+2.603 expected~+2.60  OK
2026-11-10 13:09:29 | STEP5 | sanity_TE_sep_at_throw | got=2.603 exp=2.6 | PASS
  RB: n_players=18 (expected ~18)
    t_release_rel_capped   got=+0.850 expected~+0.85  OK
2026-11-10 13:09:29 | STEP5 | sanity_RB_t_release_rel_capped | got=0.850 exp=0.85 | PASS
    depth_gain_at_throw    got=-1.512 expected~-1.50  OK
2026-11-10 13:09:29 | STEP5 | sanity_RB_depth_gain_at_throw | got=-1.512 exp=-1.5 | PASS
    sep_at_throw           got=+1.777 expected~+1.80  OK
2026-11-10 13:09:29 | STEP5 | sanity_RB_sep_at_throw | got=1.777 exp=1.8 | PASS

STEP 5: CHIP TAX & LE

In [11]:
# One-line plain-English takeaway per summary row
def phrase(r):
    m=r.metric
    if m=="t_release_rel_capped": unit="s later release" if r.effect>0 else "s earlier release"
    elif m=="depth_gain_at_throw": unit="yd depth gained vs clean" 
    elif m=="sep_at_throw": unit="yd separation vs clean"
    else: unit="release-rate diff vs clean"
    return f"[{r.comparison}/{r.position_group}] {m}: {r.effect:+.2f} {unit} (n_players={r.n_players}, n_routes={r.n_routes})"
for _,r in summary.iterrows():
    print(phrase(r))


[chip/TE] t_release_rel_capped: +1.30 s later release (n_players=22.0, n_routes=334)
[chip/TE] depth_gain_at_throw: -3.78 yd depth gained vs clean (n_players=22.0, n_routes=334)
[chip/TE] sep_at_throw: +2.60 yd separation vs clean (n_players=22.0, n_routes=334)
[chip/TE] released_rel_rate: -0.22 release-rate diff vs clean (n_players=nan, n_routes=645)
[chip/RB] t_release_rel_capped: +0.85 s later release (n_players=18.0, n_routes=244)
[chip/RB] depth_gain_at_throw: -1.51 yd depth gained vs clean (n_players=18.0, n_routes=244)
[chip/RB] sep_at_throw: +1.78 yd separation vs clean (n_players=18.0, n_routes=244)
[chip/RB] released_rel_rate: -0.14 release-rate diff vs clean (n_players=nan, n_routes=515)
[leak/TE] t_release_rel_capped: +1.50 s later release (n_players=20.0, n_routes=150)
[leak/TE] depth_gain_at_throw: -5.40 yd depth gained vs clean (n_players=20.0, n_routes=150)
[leak/TE] sep_at_throw: +2.71 yd separation vs clean (n_players=20.0, n_routes=150)
[leak/TE] released_rel_rate: -

## STEP 6 — Player table

In [12]:
# Build one row per player: chip & leak metrics gated by MIN_ROUTES_PER_GROUP (leak TE uses MIN_LEAK_ROUTES)
def player_counts(df, grp): return df[df.group==grp].groupby("nflId").size()

rows=[]
for nfl_id, g in full.groupby("nflId"):
    disp = g["displayName"].iloc[0]; pos = g["officialPosition"].iloc[0]; pg = g["position_group"].iloc[0]
    n_chip = int((g.group=="chip").sum()); n_leak=int((g.group=="leak").sum()); n_clean=int((g.group=="clean").sum())
    chip_release=chip_depth=leak_sep=np.nan
    clean_g = g[g.group=="clean"]
    # chip metrics (need >=10 chip and >=10 clean, non-NaN)
    cc = g[g.group=="chip"]
    if cc.dropna(subset=["t_release_rel_capped"]).shape[0]>=config.MIN_ROUTES_PER_GROUP and \
       clean_g.dropna(subset=["t_release_rel_capped"]).shape[0]>=config.MIN_ROUTES_PER_GROUP:
        chip_release = float(cc["t_release_rel_capped"].median()-clean_g["t_release_rel_capped"].median())
        chip_depth = float(cc["depth_gain_at_throw"].median()-clean_g["depth_gain_at_throw"].median())
    # leak metric: TE only, leak>=5 and clean>=10
    lk = g[g.group=="leak"]
    if pg=="TE" and lk.dropna(subset=["sep_at_throw"]).shape[0]>=config.MIN_LEAK_ROUTES and \
       clean_g.dropna(subset=["sep_at_throw"]).shape[0]>=config.MIN_ROUTES_PER_GROUP:
        leak_sep = float(lk["sep_at_throw"].median()-clean_g["sep_at_throw"].median())
    rows.append(dict(nflId=nfl_id, display_name=disp, official_position=pos,
                     position_group=pg,
                     n_chip=n_chip, n_leak=n_leak, n_clean=n_clean,
                     chip_release_delay_s=chip_release, chip_depth_change_yd=chip_depth,
                     leak_sep_gain_yd=leak_sep))
player_tbl = (pd.DataFrame(rows)
              .sort_values(["position_group","chip_release_delay_s"],
                           ascending=[True, False], na_position="last")
              .reset_index(drop=True))
player_tbl.to_csv(config.OUTPUT_DIR / "player_dual_duty.csv", index=False)
print("saved outputs/player_dual_duty.csv, rows:", len(player_tbl))

checks=[]
dup = player_tbl["nflId"].duplicated().any()
checks.append(qa("STEP6","no_duplicate_nflId", str(not dup), not dup))
bad_pos = player_tbl[~player_tbl.official_position.isin(config.TE_RB_POSITIONS)]
checks.append(qa("STEP6","only_te_rb_fb", f"offenders={len(bad_pos)}", len(bad_pos)==0))
# median of chip_release_delay_s (over players with a value) == Step5 TE+RB? Spec: equals Step5 chip effect.
# Step 5 effect is per position group; the player table pools both. Compare the pooled player-table median
# to the pooled within-player median computed the same way.
med_tbl = player_tbl["chip_release_delay_s"].median()
pooled_pd = []
for pg in ["TE","RB"]:
    sub=full[full.position_group==pg]
    r=an.within_player_effect(sub[sub.group=="chip"],sub[sub.group=="clean"],"t_release_rel_capped",
                              config.MIN_ROUTES_PER_GROUP,config.MIN_ROUTES_PER_GROUP,config.N_BOOTSTRAP,config.RANDOM_SEED)
    pooled_pd.append(r["player_diff"])
pooled_median = float(pd.concat(pooled_pd).median())
match = abs(med_tbl-pooled_median)<=1e-9
checks.append(qa("STEP6","chip_release_median_eq_step5_pooled", f"tbl={med_tbl:.4f} step5pooled={pooled_median:.4f}", match))
print(f"\nplayer-table median chip_release_delay_s={med_tbl:.4f} vs Step5 pooled player_diff median={pooled_median:.4f}")
for _pg in ["TE","RB"]:
    _sub = player_tbl[(player_tbl.position_group==_pg) & (player_tbl.chip_release_delay_s.notna())]
    print(f"\n[{_pg}] TOP 5 chip-tax (latest release):"); print(_sub.head(5).to_string(index=False))
    print(f"[{_pg}] BOTTOM 5 chip-tax (earliest release):"); print(_sub.tail(5).to_string(index=False))
print(f"\nSTEP 6: PLAYER TABLE | {'PASS' if all(checks) else 'FAIL'} | {len(player_tbl)} players")
assert all(checks), "STEP 6 STOP"


saved outputs/player_dual_duty.csv, rows: 256
2026-11-10 13:09:30 | STEP6 | no_duplicate_nflId | True | PASS
2026-11-10 13:09:30 | STEP6 | only_te_rb_fb | offenders=0 | PASS
2026-11-10 13:09:30 | STEP6 | chip_release_median_eq_step5_pooled | tbl=1.0500 step5pooled=1.0500 | PASS

player-table median chip_release_delay_s=1.0500 vs Step5 pooled player_diff median=1.0500

[TE] TOP 5 chip-tax (latest release):
 nflId      display_name official_position position_group  n_chip  n_leak  n_clean  chip_release_delay_s  chip_depth_change_yd  leak_sep_gain_yd
 45355   Anthony Firkser                TE             TE      10      19       70                  1.75                -5.580          6.353178
 45635     Robert Tonyan                TE             TE      12       4      123                  1.70                -4.545               NaN
 45481 Ricky Seals-Jones                TE             TE      20       7      132                  1.70                -6.100          2.355958
 52451     

## TARGET ANALYSIS (additive)

Parses the targeted receiver from `plays.playDescription` and asks: when a chipper/leaker is actually thrown to, how does his completion rate and yardage compare to clean routes? This cell block is additive and does not modify any numbers above.

In [13]:
import re
# 1. parse target from playDescription
TARGET_RE = re.compile(
    r"pass (?:incomplete )?(?:short|deep) (?:left|middle|right) (?:to|intended for) "
    r"([A-Z][a-zA-Z]*\.\s?[A-Za-z'\.\- ]+?)(?:\s(?:to|for|pushed|ran|\[|\.|,|$)|\.$| \()")

def parse_target(desc):
    if not isinstance(desc, str):
        return None
    m = TARGET_RE.search(desc)
    return m.group(1).strip() if m else None

# plays.csv is not loaded elsewhere in this notebook; load it here (read-only)
plays = pd.read_csv(config.DATA_DIR / "plays.csv")
plays_t = plays.copy()
plays_t["parsed_target"] = plays_t["playDescription"].apply(parse_target)

# 2. abbreviated route-player name: first initial + '.' + rest of displayName
def abbr_name(name):
    parts = name.split(" ", 1)
    return name if len(parts) < 2 else parts[0][0] + "." + parts[1]

def _norm(s):
    return s.replace(" ", "").lower() if isinstance(s, str) else s

tgt = full.copy()
tgt["abbr_name"] = tgt["displayName"].apply(abbr_name)
tgt = tgt.merge(
    plays_t[["gameId","playId","parsed_target","passResult","prePenaltyPlayResult"]],
    on=["gameId","playId"], how="left")
tgt["targeted"] = [(_norm(a) == _norm(t)) if isinstance(t, str) else False
                   for a, t in zip(tgt["abbr_name"], tgt["parsed_target"])]
print("merged target frame rows:", len(tgt))


merged target frame rows: 14435


In [14]:
# 3. CHECKS
# parse rate on throws (C/I/IN)
_throws = plays_t[plays_t["passResult"].isin(["C","I","IN"])]
parse_rate = _throws["parsed_target"].notna().mean()
qa("STEP-TGT", "parse_rate_on_throws", f"{parse_rate:.3f}", 0.88 <= parse_rate <= 0.94)

# 10 random parsed descriptions vs matched displayName (eyeball)
print("\n10 random targeted routes (parsed target vs route displayName):")
_s = tgt[tgt.targeted].sample(10, random_state=config.RANDOM_SEED)
for _, r in _s.iterrows():
    print(f"  parsed={r['parsed_target']!r:20s} -> displayName={r['displayName']!r}")

# targeted route counts
_n_tgt = int(tgt["targeted"].sum())
_by_grp = tgt[tgt.targeted].groupby("group").size().to_dict()
print(f"\nTE/RB targeted routes total: {_n_tgt} (expect ~2665)")
print("by group:", _by_grp, "(expect chip ~157, leak ~81)")
qa("STEP-TGT", "targeted_routes_total", f"{_n_tgt}", 2500 <= _n_tgt <= 2800)
qa("STEP-TGT", "targeted_chip_count", f"{_by_grp.get('chip')}", 140 <= _by_grp.get('chip',0) <= 175)
qa("STEP-TGT", "targeted_leak_count", f"{_by_grp.get('leak')}", 70 <= _by_grp.get('leak',0) <= 92)


2026-11-10 13:09:30 | STEP-TGT | parse_rate_on_throws | 0.913 | PASS

10 random targeted routes (parsed target vs route displayName):
  parsed='C.Uzomah'           -> displayName='C.J. Uzomah'
  parsed='C.Ham'              -> displayName='C.J. Ham'
  parsed='D.Schultz'          -> displayName='Dalton Schultz'
  parsed='C.Hyde'             -> displayName='Carlos Hyde'
  parsed='C.Brate'            -> displayName='Cameron Brate'
  parsed='M.Pruitt'           -> displayName='MyCole Pruitt'
  parsed='M.Alie-Cox'         -> displayName='Mo Alie-Cox'
  parsed='E.Ebron'            -> displayName='Eric Ebron'
  parsed='J.Robinson'         -> displayName='James Robinson'
  parsed='K.Hunt'             -> displayName='Kareem Hunt'

TE/RB targeted routes total: 2664 (expect ~2665)
by group: {'chip': 157, 'clean': 2426, 'leak': 81} (expect chip ~157, leak ~81)
2026-11-10 13:09:30 | STEP-TGT | targeted_routes_total | 2664 | PASS
2026-11-10 13:09:30 | STEP-TGT | targeted_chip_count | 157 | PASS
2026-

True

In [15]:
# 4-5. target_summary: exclude sacks, use prePenaltyPlayResult for yards, split TE vs RB
_t = tgt[tgt.targeted & (tgt.passResult != "S") & tgt.position_group.notna()].copy()
_t["complete"] = (_t["passResult"] == "C").astype(int)

_rows = []
for (grp, pg), d in _t.groupby(["group", "position_group"]):
    _rows.append(dict(group=grp, position_group=pg, n_targets=len(d),
                      completion_rate=round(d["complete"].mean(), 4),
                      mean_yards=round(d["prePenaltyPlayResult"].mean(), 3),
                      median_yards=round(d["prePenaltyPlayResult"].median(), 3)))
target_summary = pd.DataFrame(_rows).sort_values(["position_group","group"]).reset_index(drop=True)
target_summary.to_csv(config.OUTPUT_DIR / "target_summary.csv", index=False)
print("saved outputs/target_summary.csv")
print(target_summary.to_string(index=False))

# README numbers: chip vs clean when targeted (pooled TE+RB)
_chip = _t[_t.group == "chip"]; _clean = _t[_t.group == "clean"]
_chip_comp = _chip["complete"].mean(); _clean_comp = _clean["complete"].mean()
_chip_yds = _chip["prePenaltyPlayResult"].mean(); _clean_yds = _clean["prePenaltyPlayResult"].mean()
qa("STEP-TGT", "readme_chip_vs_clean",
   f"comp {_chip_comp*100:.0f}% vs {_clean_comp*100:.0f}%, yds {_chip_yds:.1f} vs {_clean_yds:.1f}", True)
print(f"\nREADME: chippers complete {_chip_comp*100:.0f}% vs {_clean_comp*100:.0f}% on clean, "
      f"gain {_clean_yds-_chip_yds:.1f} fewer yards (chip {_chip_yds:.1f} vs clean {_clean_yds:.1f}).")
print("\nTARGET ANALYSIS | done (additive, no existing numbers changed)")


saved outputs/target_summary.csv
group position_group  n_targets  completion_rate  mean_yards  median_yards
 chip             RB         60           0.8333       5.533           4.5
clean             RB       1124           0.7696       6.073           5.0
 leak             RB          8           0.8750       9.000           8.0
 chip             TE         97           0.8763       6.392           6.0
clean             TE       1302           0.6905       7.980           6.0
 leak             TE         73           0.7260       6.397           6.0
2026-11-10 13:09:30 | STEP-TGT | readme_chip_vs_clean | comp 86% vs 73%, yds 6.1 vs 7.1 | PASS

README: chippers complete 86% vs 73% on clean, gain 1.0 fewer yards (chip 6.1 vs clean 7.1).

TARGET ANALYSIS | done (additive, no existing numbers changed)
